# OLC overlap framework notes

这个 notebook 是浏览测试版：每个段落、表格、公式都拆成独立 cell，尽量避免在普通 JSON 视图里出现大量反斜杠 n。

## 1. 图处理

记录从 reads 到 overlap graph 的构造、筛边、压缩和可恢复性评估。

<table><thead><tr><th>阶段</th><th>当前实现</th><th>需要记录的内容</th></tr></thead><tbody><tr><td>read simulation</td><td><code>RandomReadSimulator</code></td><td>genome length, read length, step, error rates, GC fraction</td></tr><tr><td>candidate finding</td><td><code>Minimap2CandidateFinder</code></td><td>minimap2 preset, min overlap, error hint, overhang tolerance</td></tr><tr><td>refinement</td><td><code>ParasailOverlapRefiner</code></td><td>min overlap, max error, margin, DP scoring</td></tr><tr><td>edge scoring</td><td><code>AlignmentMIScorer</code>, <code>OverlapRewardScorer</code></td><td>overlap length, DP score, MI/NMI, quality score</td></tr></tbody></table>

重要路径：<code>candidate_finder.py</code> 解析 minimap2 PAF；<code>refiner.py</code> 做 Parasail semi-global refinement；<code>evaluator.py</code> 给边贴 adjacent / jump / wrong 标签；<code>overlap_features.py</code> 计算 GC 和 overlap composition。

### 图质量指标

<table><thead><tr><th>指标</th><th>含义</th><th>备注</th></tr></thead><tbody><tr><td>candidate edges</td><td>进入 Hamiltonian 的候选边数</td><td>直接影响变量数</td></tr><tr><td>adjacent / jump / wrong</td><td>候选边 truth label</td><td>jump 不一定坏，但会影响布局</td></tr><tr><td>Hamiltonian path available</td><td>真路径是否完全存在于候选图</td><td>若不存在，则不能期待严格 ground hit</td></tr><tr><td>full DNA recoverable</td><td>选出的边是否仍可恢复完整 DNA/span</td><td>比 exact ground hit 更宽松</td></tr><tr><td>prunable extra edges</td><td>多余边是否可剪掉</td><td>用于判断 path-cover/cycle 解是否可后处理</td></tr></tbody></table>

## 2. 哈密顿量

本节记录目前实现和测试过的 QUBO Hamiltonian。符号约定：reads 数为 <code>N</code>，候选有向 overlap 边集合为 <code>E</code>；<code>x_{v,j}</code> 表示 read <code>v</code> 放在位置 <code>j</code>；<code>y_{uv}</code> 表示选择有向边 <code>u -> v</code>；<code>r_{uv}</code> 是归一化或原始 overlap reward。

<table><thead><tr><th>模型</th><th>变量</th><th>核心想法</th><th>当前结论</th></tr></thead><tbody><tr><td><code>MissingEdgeQUBOHamiltonian</code></td><td><code>x[v,j]</code></td><td>一热排列 + 缺边惩罚</td><td>小规模基线，适合验证约束</td></tr><tr><td><code>WeightedOverlapQUBOHamiltonian</code></td><td><code>x[v,j]</code></td><td>一热排列 + 缺边惩罚 + overlap reward</td><td>能进合法排列，但容易卡在差排列</td></tr><tr><td><code>EdgePathDAGQUBOHamiltonian</code></td><td><code>y[u,v]</code></td><td>选择 <code>N-1</code> 条边形成 Hamiltonian path</td><td>35 reads 很稳，300+ 变量后变难</td></tr><tr><td><code>EdgePathCoverDAGQUBOHamiltonian</code></td><td><code>y[u,v]</code> + source/sink</td><td>允许 path cover，再用 path count 项逼近单路径</td><td>power reward 后改善，100 reads 可恢复性好</td></tr><tr><td><code>EdgeCycleCoverDAGQUBOHamiltonian</code></td><td><code>E + 2N</code></td><td>加 void 节点，把线性路径表达为 void-cycle</td><td>100 reads power2 测试最稳之一</td></tr><tr><td><code>PDFAssemblyQUBOHamiltonian</code></td><td><code>y[u,v]</code></td><td>path A + PDF 的长度/GC/MI 项</td><td><code>A+B</code> 长度项是最新最强线索</td></tr></tbody></table>

### 2.1 MissingEdgeQUBOHamiltonian

$$E = A_r\sum_v\left(1-\sum_j x_{v,j}\right)^2 + A_p\sum_j\left(1-\sum_v x_{v,j}\right)^2 + B_m\sum_{j=0}^{N-2}\sum_{(u,v)\in E^c} x_{u,j}x_{v,j+1}$$

<p><b>含义</b></p><table><thead><tr><th>部分</th><th>作用</th><th>读法</th></tr></thead><tbody><tr><td><code>x[v,j]</code></td><td>位置变量</td><td>read <code>v</code> 是否放在第 <code>j</code> 个位置</td></tr><tr><td><code>A_r</code> 项</td><td>read one-hot</td><td>每个 read 必须且只能出现一次</td></tr><tr><td><code>A_p</code> 项</td><td>position one-hot</td><td>每个位置必须且只能放一个 read</td></tr><tr><td><code>B_m</code> 项</td><td>缺边惩罚</td><td>若相邻两个位置没有候选 overlap 边，则加罚</td></tr><tr><td>局限</td><td>只看边存在性</td><td>不能区分长 overlap、短 overlap、MI 高低或 alignment 质量</td></tr></tbody></table>

<p><b>测试结果</b></p><table><thead><tr><th>测试</th><th>结果</th><th>基底态结论</th></tr></thead><tbody><tr><td>3-read toy chain</td><td>合法路径能量为 0，solver 恢复 <code>r0 -> r1 -> r2</code></td><td>小玩具例子可以达到基底</td></tr><tr><td>大规模 sweep</td><td>目前没有作为主力模型系统测试</td><td>没有证据表明可扩展到较大 reads</td></tr></tbody></table><p><b>一句话结论</b>：适合作为 assignment 约束是否写对的基线，不适合作为当前主攻模型。</p>

### 2.2 WeightedOverlapQUBOHamiltonian

$$E = A_r\sum_v\left(1-\sum_j x_{v,j}\right)^2 + A_p\sum_j\left(1-\sum_v x_{v,j}\right)^2 + \sum_{j=0}^{N-2}\left(C_m\sum_{(u,v)\in E^c}x_{u,j}x_{v,j+1} - B_e\sum_{(u,v)\in E} r_{uv}x_{u,j}x_{v,j+1}\right)$$

<p><b>含义</b></p><table><thead><tr><th>部分</th><th>作用</th><th>读法</th></tr></thead><tbody><tr><td><code>A_r, A_p</code></td><td>排列合法性</td><td>仍然是 <code>N^2</code> 个位置变量的一热约束</td></tr><tr><td><code>C_m</code> 项</td><td>缺边惩罚</td><td>相邻位置若不是候选边，就被惩罚</td></tr><tr><td><code>B_e r_uv</code> 项</td><td>边质量奖励</td><td>候选边越好，越鼓励把两个 read 放成相邻</td></tr><tr><td><code>r_uv</code> 来源</td><td>overlap reward</td><td>可来自 overlap length、DP score、MI/NMI、mapq、matches 或 quality</td></tr><tr><td>主要风险</td><td>合法但不对</td><td>one-hot 很容易满足，但局部 reward 可能把排列推到错误邻接</td></tr></tbody></table>

<p><b>测试结果</b></p><table><thead><tr><th>测试文件/规模</th><th>结果</th><th>基底态结论</th></tr></thead><tbody><tr><td><code>sqa_parameter_sweep.csv</code></td><td>81 例中 50 例得到合法 one-hot layout，但 0 次 ground hit</td><td>不能稳定达到基底</td></tr><tr><td><code>sqa_trotter32_grid.csv</code></td><td>180 例中 173 例合法，但只有 1 次 ground hit</td><td>基底命中非常稀有</td></tr><tr><td>唯一明确命中点</td><td>约 16 reads，<code>A_r=A_p=75, C_m=60, B_e=40, trotter=32, seed=42</code>，得到 15/15 正确邻接</td><td>小规模特定参数下可以达到基底</td></tr></tbody></table><p><b>一句话结论</b>：它能很好地满足“排列合法”，但不能可靠解决“排列正确”。</p>

### 2.3 EdgePathDAGQUBOHamiltonian

$$E = A_c\left(\sum_{(u,v)\in E} y_{uv} - (N-1)\right)^2 + A_d\sum_v \sum_{u_1<u_2} y_{u_1v}y_{u_2v} + A_d\sum_v \sum_{w_1<w_2} y_{vw_1}y_{vw_2} - B\sum_{(u,v)\in E} r_{uv}y_{uv}$$

<p><b>含义</b></p><table><thead><tr><th>部分</th><th>作用</th><th>读法</th></tr></thead><tbody><tr><td><code>y[u,v]</code></td><td>边变量</td><td>只给候选 overlap 边建变量，变量数从 <code>N^2</code> 降到 <code>|E|</code></td></tr><tr><td><code>A_c</code> 项</td><td>边数约束</td><td>要求最终选出 <code>N-1</code> 条边</td></tr><tr><td><code>A_d</code> incoming 项</td><td>入度冲突惩罚</td><td>同一个 read 不能有多个 predecessor</td></tr><tr><td><code>A_d</code> outgoing 项</td><td>出度冲突惩罚</td><td>同一个 read 不能有多个 successor</td></tr><tr><td><code>B r_uv</code> 项</td><td>边质量奖励</td><td>在满足路径约束的同时偏好高 reward overlap</td></tr><tr><td>前提</td><td>候选图是 DAG</td><td>如果候选图里没有覆盖所有 reads 的 Hamiltonian path，就不应期待 exact ground hit</td></tr></tbody></table>

<p><b>测试结果</b></p><table><thead><tr><th>规模/设置</th><th>结果</th><th>基底态结论</th></tr></thead><tbody><tr><td>35 reads / 131 edge variables</td><td><code>A_c=100, A_d=120, B=20</code> 下 10/10 ground hit</td><td>小规模稳定达到基底</td></tr><tr><td>217 variables</td><td>5/5 ground hit</td><td>仍然可以达到基底</td></tr><tr><td>315 variables</td><td>最佳约 4/10 ground hit</td><td>开始明显不稳定</td></tr><tr><td>405 variables</td><td>约 2/10 ground hit</td><td>多数情况下不能达到基底</td></tr><tr><td>577 variables</td><td>未观察到稳定命中</td><td>当前参数下基本失败</td></tr><tr><td>GC=0.65，80/90/100 reads 对比</td><td>old edge-path DAG 为 0/6 ground hit</td><td>较大图上不能达到基底</td></tr></tbody></table><p><b>一句话结论</b>：这是从位置变量转向边变量的关键一步；小规模很强，但全局边数平方项在较大图上变成瓶颈。</p>

### 2.4 EdgePathCoverDAGQUBOHamiltonian

$$E = A_d\sum_v\left(1-s_v-\sum_u y_{uv}\right)^2 + A_d\sum_v\left(1-t_v-\sum_w y_{vw}\right)^2 + A_i\sum_v s_vt_v + A_k\left(\sum_v s_v-1\right)^2 - B\sum_{(u,v)\in E}r_{uv}y_{uv}$$

<p><b>含义</b></p><table><thead><tr><th>部分</th><th>作用</th><th>读法</th></tr></thead><tbody><tr><td><code>s_v</code></td><td>source marker</td><td>read <code>v</code> 是某条路径的起点</td></tr><tr><td><code>t_v</code></td><td>sink marker</td><td>read <code>v</code> 是某条路径的终点</td></tr><tr><td>incoming equation</td><td>覆盖每个 read</td><td>每个 read 要么有一个 predecessor，要么被标成 source</td></tr><tr><td>outgoing equation</td><td>覆盖每个 read</td><td>每个 read 要么有一个 successor，要么被标成 sink</td></tr><tr><td><code>A_i s_v t_v</code></td><td>孤立点惩罚</td><td>避免一个 read 同时是 source 和 sink 而不连接别人</td></tr><tr><td><code>A_k</code> 项</td><td>路径数约束</td><td>把 path cover 推向单条路径</td></tr><tr><td>建模变化</td><td>允许中间态是 cover</td><td>相比直接强制 Hamiltonian path，搜索空间更容易出现可后处理解</td></tr></tbody></table>

<p><b>测试结果</b></p><table><thead><tr><th>规模/设置</th><th>结果</th><th>基底态结论</th></tr></thead><tbody><tr><td>早期参数</td><td>经常得到合法 path cover，但不一定是单路径或 ground state</td><td>约束方向正确，但不够稳</td></tr><tr><td>70 reads / 415 variables / power2 grid</td><td>path-cover 8/16 ground hit</td><td>部分达到基底</td></tr><tr><td>100 reads / 595 variables / power2 multiseed</td><td>6/8 ground hit</td><td>多数 seed 达到基底</td></tr><tr><td>100 reads recoverability / power2</td><td>7/10 ground hit，同时 10/10 full DNA recoverable</td><td>exact ground 不是满分，但组装可恢复性满分</td></tr></tbody></table><p><b>一句话结论</b>：它不总是精确命中基底态，但经常给出可以恢复完整 DNA 的解，是从“严格路径”走向“可后处理 assembly”的重要改进。</p>

### 2.5 EdgeCycleCoverDAGQUBOHamiltonian

$$E = A_d\sum_v\left(1-s_v-\sum_u y_{uv}\right)^2 + A_d\sum_v\left(1-t_v-\sum_w y_{vw}\right)^2 + A_d\left(1-\sum_v s_v\right)^2 + A_d\left(1-\sum_v t_v\right)^2 - B\sum_{(u,v)\in E}r_{uv}y_{uv}$$

<p><b>含义</b></p><table><thead><tr><th>部分</th><th>作用</th><th>读法</th></tr></thead><tbody><tr><td><code>E + 2N</code></td><td>变量规模</td><td>候选边变量加上每个 read 的 source/sink marker</td></tr><tr><td><code>s_v</code></td><td>void 到 read</td><td>可理解为虚拟节点 <code>void -> v</code></td></tr><tr><td><code>t_v</code></td><td>read 到 void</td><td>可理解为 <code>v -> void</code></td></tr><tr><td>read degree</td><td>真实 read exactly-one</td><td>每个 read 恰好一个 incoming，恰好一个 outgoing</td></tr><tr><td>void degree</td><td>唯一入口和出口</td><td>只允许一个 start 和一个 end</td></tr><tr><td>结构解释</td><td>路径变成环</td><td>线性路径写成 <code>void -> start -> ... -> end -> void</code></td></tr></tbody></table>

<p><b>测试结果</b></p><table><thead><tr><th>规模/设置</th><th>结果</th><th>基底态结论</th></tr></thead><tbody><tr><td>100 reads / 395 candidate edges / 595 variables / power2 multiseed</td><td>cycle 模型 8/8 ground hit</td><td>稳定达到基底</td></tr><tr><td>100 reads recoverability / power1</td><td>7/10 ground hit，10/10 recoverable</td><td>基底不满分，但可恢复性满分</td></tr><tr><td>100 reads recoverability / power2</td><td>10/10 ground hit，10/10 valid，10/10 recoverable</td><td>完全达到基底，且组装可恢复</td></tr></tbody></table><p><b>一句话结论</b>：这是 100-read 规模最稳定的模型之一；power2 reward 下目前表现非常干净。</p>

### 2.6 PDFAssemblyQUBOHamiltonian

$$E = A\left(\sum_{(a,b)} z_{ab}-(N-1)\right)^2 + A\cdot degree\_conflicts + B\left(\sum_i l_i - \sum_{(a,b)}O_{ab}z_{ab} - L_{target}\right)^2 + C\left[\left(\sum_i g_i - \sum_{(a,b)}G^{ov}_{ab}z_{ab}\right)-p_{GC}\left(\sum_i l_i - \sum_{(a,b)}O_{ab}z_{ab}\right)\right]^2 - D\sum_{(a,b)}I_{ab}z_{ab}$$

<p><b>含义</b></p><table><thead><tr><th>部分</th><th>作用</th><th>读法</th></tr></thead><tbody><tr><td><code>z_ab</code></td><td>边变量</td><td>与 <code>y[u,v]</code> 类似，表示是否选择 overlap 边 <code>a -> b</code></td></tr><tr><td><code>A</code> path 项</td><td>路径骨架</td><td>当前实现沿用 edge-path 的 <code>N-1</code> 边数约束和 degree conflict 约束</td></tr><tr><td><code>B</code> 长度项</td><td>assembly length prior</td><td>让 read 总长度减去被 overlap 抵消的长度后，接近目标 genome length</td></tr><tr><td><code>C</code> GC 项</td><td>composition prior</td><td>让组装后的 GC fraction 接近目标 <code>p_GC</code></td></tr><tr><td><code>D</code> 信息项</td><td>边信息量奖励</td><td>奖励 MI/NMI 等 overlap 信息量高的边</td></tr><tr><td>当前实现</td><td>无 source/sink 变量</td><td>它不是 path-cover/cycle-cover，而是 edge-path 骨架加 PDF 统计项</td></tr></tbody></table>

<p><b>测试结果</b></p><table><thead><tr><th>测试文件/设置</th><th>结果</th><th>基底态结论</th></tr></thead><tbody><tr><td><code>pdf_vs_edge_path_ablation_35_40_45.csv</code></td><td><code>pdf_A_B</code> 为 6/6 ground hit；<code>pdf_A_B_C_D</code> 也是 6/6；<code>A_only</code>、<code>A_C</code>、<code>A_D</code> 为 0/6</td><td><code>B</code> 长度项是关键，单独 <code>A</code>/<code>C</code>/<code>D</code> 不能达到基底</td></tr><tr><td><code>pdf_vs_edge_path_optimized_gc065_80_90_100.csv</code></td><td><code>pdf_A_B</code> 在 80/90/100 reads 上 6/6 ground hit；old edge-path DAG 为 0/6</td><td>在较大 read 数上，PDF 长度项显著优于旧 edge-path</td></tr><tr><td>70 reads raw MI error sweep</td><td>2%、3%、5% 错误率 grid 分别为 25/25、25/25、15/15 ground hit</td><td>raw MI 下错误率到 5% 仍能达到基底</td></tr><tr><td>70 reads NMI error sweep</td><td>2%、3%、5% 错误率 grid 分别为 25/25、25/25、14/15 ground hit；5% NMI 仍 15/15 full DNA recoverable</td><td>NMI 下 exact ground hit 略降，但可恢复性仍满分</td></tr></tbody></table><p><b>一句话结论</b>：这是目前最新、最强的方向；实验证据显示 <code>B</code> 长度项提供了决定性约束，<code>C</code>/<code>D</code> 更像辅助项而不是主因。</p>